# Атрибуты товаров: пересечение и распределение по категориям (polars)

In [4]:
import polars as pl
import json

df = pl.read_parquet('data/items_human.parquet')
df.head()

id,name,attributes,category
i64,str,str,str
197,"""victor reinz п…","""{""артикул"":""70…","""Автотовары"""
415,"""stellox диск т…","""{""артикул"":""st…","""Автотовары"""
427,"""комплект подши…","""{""артикул прои…","""Автотовары"""
1027,"""kraft подшипни…","""{""артикул"":""11…","""Автотовары"""
2639,"""фильтр салонны…","""{""альтернативн…","""Автотовары"""


## 1. Атрибуты, общие для всех товаров (без учёта категории)

In [5]:
common = None
for attrs_str in df['attributes']:
    keys = set(json.loads(attrs_str).keys())
    common = keys if common is None else common & keys

print(common)

set()


Пусто — общего для вообще всех товаров атрибута нет, т.к. категории очень разные.

## 2. Распределение атрибутов по категориям

Для каждой категории считаем, у какой доли товаров этой категории встречается каждый атрибут.

In [10]:
exploded = (
    df
    .with_columns(
        pl.col('attributes')
        .map_elements(lambda s: list(json.loads(s).keys()), return_dtype=pl.List(pl.Utf8))
        .alias('keys')
    )
    .explode('keys')
)

category_sizes = df.group_by('category').len().rename({'len': 'total'})

dist = (
    exploded
    .group_by(['category', 'keys'])
    .len()
    .join(category_sizes, on='category')
    .with_columns((pl.col('len') / pl.col('total')).alias('pct'))
    .sort(['category', 'pct'], descending=[False, True])
)

dist

category,keys,len,total,pct
str,str,u32,u32,f64
"""Автотовары""","""тип""",24473,37752,0.648257
"""Автотовары""","""бренд""",23481,37752,0.62198
"""Автотовары""","""партномер (арт…",16130,37752,0.427262
"""Автотовары""","""комплектация""",14387,37752,0.381092
"""Автотовары""","""артикул""",10136,37752,0.268489
"""Автотовары""","""артикул произв…",9253,37752,0.2451
"""Автотовары""","""вид техники""",8901,37752,0.235776
"""Автотовары""","""страна-изготов…",8863,37752,0.234769
"""Автотовары""","""oem-номер""",8166,37752,0.216306


## 3. Топ-10 атрибутов для каждой категории

In [11]:
top10 = dist.group_by('category', maintain_order=False).head(10).sort(['category', 'pct'], descending=[False, True])

for category in df['category'].unique().sort():
    print(f"\n=== {category} ===")
    for row in top10.filter(pl.col('category') == category).iter_rows(named=True):
        print(f"  {row['pct']:.1%}\t{row['keys']}")


=== Автотовары ===
  64.8%	тип
  62.2%	бренд
  42.7%	партномер (артикул производителя)
  38.1%	комплектация
  26.8%	артикул
  24.5%	артикул производителя
  23.6%	вид техники
  23.5%	страна-изготовитель
  21.6%	oem-номер
  20.5%	цвет товара

=== Аптека ===
  60.1%	пол
  57.7%	оптическая сила
  55.6%	форма оправы
  55.5%	материал линз
  54.6%	бренд
  54.1%	тип
  52.4%	покрытие линз
  48.5%	страна-изготовитель
  43.0%	цвет товара
  42.3%	комплектация

=== Бытовая техника ===
  64.1%	тип
  57.1%	бренд
  56.0%	гарантийный срок
  44.1%	комплектация
  41.9%	цвет товара
  39.4%	страна-изготовитель
  29.2%	тип управления
  24.7%	размеры, мм
  24.2%	модель
  22.5%	валюта

=== Бытовая химия ===
  61.4%	бренд
  55.7%	тип
  50.8%	состав
  48.2%	комплектация
  39.6%	страна-изготовитель
  32.8%	валюта
  31.3%	единиц в одном товаре
  29.0%	страна производства
  25.4%	длина упаковки
  25.3%	высота упаковки

=== Галантерея и аксессуары ===
  61.7%	пол
  59.6%	тип
  55.8%	комплектация
  54.0%	бренд
  53

## 4. Атрибуты, общие для всех товаров внутри каждой категории (100%)

In [9]:
always_present = dist.filter(pl.col('pct') == 1.0).sort('category')
always_present

category,keys,len,total,pct
str,str,u32,u32,f64
